# Taylor Approximation Tests: F0 – F3

Verifies the second-order Taylor expansion for each functional in the `RecNFP` cascade:
$$f(x + dx) = f(x) + df(x, dx) + \tfrac{1}{2}\,d^2f(x, dx, dx) + O(\|dx\|^3)$$

**Expected:** e1 ratio ≈ 4, e2 ratio ≈ 8 when $\|dx\|$ is halved.

| Functional | Nonlinear component tested | Expected |
|---|---|---|
| F3 | $x_{22} = \mathcal{S}_{x_{33}}(x_{32})$ — B-spline shift of proj by pos | e2 ~ O(h³) |
| F2 | $x_{12} = e^{i x_{22}}$ — phase encoding | e2 ~ O(h³) |
| F1 | $x_0 = D(x_{11} \cdot x_{12})$ — bilinear | **e2 = 0** |
| F0 | $\frac{1}{N}\||x_0| - d\|_2^2$ — data fidelity | e2 ~ O(h³) |

In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from mpi4py import MPI
from types import SimpleNamespace
from holotomocupy.rec_nfp_mpi import RecNFP

n      = 128
nz     = 128
nobj   = 192
nzobj  = 192
ntheta = 8

args = SimpleNamespace(
    energy                  = 17.1,
    detector_pixelsize      = 1.4760147601476e-6 * 8,
    focustodetectordistance = 1.217,
    z1                      = 5.110e-3,
    ntheta                  = ntheta,
    nz                      = nz,
    n                       = n,
    nzobj                   = nzobj,
    nobj                    = nobj,
    obj_dtype               = 'float32',
    rho                     = [1, 1, 1],
    niter                   = 1,
    nchunk                  = 4,
    vis_step                = -1,
    err_step                = -1,
    start_iter              = 0,
    comm                    = MPI.COMM_WORLD,
)

cl = RecNFP(args)

In [ ]:
rng = np.random.default_rng(42)

def rc(shape, scale=1.0):
    return (rng.standard_normal(shape) + 1j*rng.standard_normal(shape)).astype('complex64') * scale

def rf(shape, scale=1.0):
    return (rng.standard_normal(shape) * scale).astype('float32')

# Evaluation points
prb_g  = cp.array(rc((nz, n))) + 1
proj_g = cp.array(rf((nzobj, nobj), 0.1))
pos_g  = cp.array(rf((ntheta, 2), 0.5))
data_g = cp.array(np.abs(rf((ntheta, nz, n))) + 1)

# Perturbation directions
dprb_g  = cp.array(rc((nz, n)))
dproj_g = cp.array(rf((nzobj, nobj), 0.1))
dpos_g  = cp.array(rf((ntheta, 2)))

L = np.linspace(0, 0.1, 20, dtype='float32')

## F3: $(prb,\,proj,\,pos) \mapsto (prb,\; \mathcal{S}_{pos}(proj))$
Tests the 2nd component: B-spline shift of proj by per-angle positions.

In [ ]:
x3    = [prb_g, proj_g, pos_g]
dw0_3 = [dprb_g, dproj_g, dpos_g]

err1, err2, err3 = np.zeros(20), np.zeros(20), np.zeros(20)
f_w = cl.F3(x3)[1]
for k, l in enumerate(L):
    dx3 = [l*dv for dv in dw0_3]
    a   = cl.F3([v + dv for v, dv in zip(x3, dx3)])[1]
    df  = cl.dF3(x3, dx3, return_x=False)[1]
    d2f = cl.d2F_dF3(x3, dx3, dx3, [None]*3)[1]
    err1[k] = float(cp.linalg.norm(f_w - a))
    err2[k] = float(cp.linalg.norm(f_w + df - a))
    err3[k] = float(cp.linalg.norm(f_w + df + 0.5*d2f - a))

plt.semilogy(L, err1, label='O(h)'); plt.semilogy(L, err2, label='O(h²)'); plt.semilogy(L, err3, label='O(h³)')
plt.legend(); plt.grid(); plt.title('F3 approximation'); plt.show()
print('e2/e1 ratios (expect ~4):', np.round(err1[10:15] / err2[10:15], 2))
print('e3/e2 ratios (expect ~8):', np.round(err2[10:15] / err3[10:15], 2))

## F2: $(prb,\,shifted\_proj) \mapsto (prb,\; e^{i\cdot shifted\_proj})$
Tests the 2nd component: phase encoding.

In [ ]:
x22_f2  = cp.array(rf((ntheta, nz, n), 0.1))
dx22_f2 = cp.array(rf((ntheta, nz, n), 0.1))

x2    = [prb_g, x22_f2]
dw0_2 = [dprb_g, dx22_f2]

err1, err2, err3 = np.zeros(20), np.zeros(20), np.zeros(20)
f_w = cl.F2(x2)[1]
for k, l in enumerate(L):
    dx2 = [l*dv for dv in dw0_2]
    a   = cl.F2([v + dv for v, dv in zip(x2, dx2)])[1]
    df  = cl.dF2(x2, dx2, return_x=False)[1]
    d2f = cl.d2F_dF2(x2, dx2, dx2, [None, None])[1]
    err1[k] = float(cp.linalg.norm(f_w - a))
    err2[k] = float(cp.linalg.norm(f_w + df - a))
    err3[k] = float(cp.linalg.norm(f_w + df + 0.5*d2f - a))

plt.semilogy(L, err1, label='O(h)'); plt.semilogy(L, err2, label='O(h²)'); plt.semilogy(L, err3, label='O(h³)')
plt.legend(); plt.grid(); plt.title('F2 approximation'); plt.show()
print('e2/e1 ratios (expect ~4):', np.round(err1[10:15] / err2[10:15], 2))
print('e3/e2 ratios (expect ~8):', np.round(err2[10:15] / err3[10:15], 2))

## F1: $(prb,\, exp\_proj) \mapsto D(prb \cdot exp\_proj)$
Bilinear in $(prb,\, exp\_proj)$ — **e2 at machine precision**.

In [ ]:
x11_f1  = cp.array(rc((ntheta, nz, n)))
x12_f1  = cp.array(rc((ntheta, nz, n)))
dx11_f1 = cp.array(rc((ntheta, nz, n), 0.1))
dx12_f1 = cp.array(rc((ntheta, nz, n), 0.1))

x1    = [x11_f1, x12_f1]
dw0_1 = [dx11_f1, dx12_f1]

err1, err2, err3 = np.zeros(20), np.zeros(20), np.zeros(20)
f_w = cl.F1(x1)
for k, l in enumerate(L):
    dx1 = [l*dv for dv in dw0_1]
    a   = cl.F1([v + dv for v, dv in zip(x1, dx1)])
    df  = cl.dF1(x1, dx1, return_x=False)
    d2f = cl.d2F_dF1(x1, dx1, dx1, [None, None])
    err1[k] = float(cp.linalg.norm(f_w - a))
    err2[k] = float(cp.linalg.norm(f_w + df - a))
    err3[k] = float(cp.linalg.norm(f_w + df + 0.5*d2f - a))

plt.semilogy(L, err1, label='O(h)'); plt.semilogy(L, err2, label='O(h²)'); plt.semilogy(L[1:], err3[1:], label='O(h³) ~ 0')
plt.legend(); plt.grid(); plt.title('F1 approximation (bilinear: e3 ≈ 0)'); plt.show()
print('e2/e1 ratios (expect ~4):', np.round(err1[10:15] / err2[10:15], 2))
print('e3 (expect ~0):',           np.round(err3[10:15], 3))

## F0: $\frac{1}{N}\||x_0| - d\|_2^2$

In [ ]:
x0_f0  = cp.array(rc((ntheta, nz, n)))
dx0_f0 = cp.array(rc((ntheta, nz, n), 0.1))

err1, err2, err3 = np.zeros(20), np.zeros(20), np.zeros(20)
f_w = float(cl.F0(x0_f0, data_g))
for k, l in enumerate(L):
    dx  = l * dx0_f0
    a   = float(cl.F0(x0_f0 + dx, data_g))
    df  = float(cl.dF0(x0_f0, dx, data_g))
    d2f = float(cl.d2F_dF0(x0_f0, dx, dx, None, data_g))
    err1[k] = abs(f_w - a)
    err2[k] = abs(f_w + df - a)
    err3[k] = abs(f_w + df + 0.5*d2f - a)

plt.semilogy(L, err1, label='O(h)'); plt.semilogy(L, err2, label='O(h²)'); plt.semilogy(L, err3, label='O(h³)')
plt.legend(); plt.grid(); plt.title('F0 approximation'); plt.show()
print('e2/e1 ratios (expect ~4):', np.round(err1[10:15] / err2[10:15], 2))
print('e3/e2 ratios (expect ~8):', np.round(err2[10:15] / err3[10:15], 2))